In [ ]:
!pip install -q crewai groq gradio diffusers torch transformers
!pip install -q evaluate rouge_score

In [ ]:
!pip install -q crewai groq gradio diffusers torch transformers
!pip install -q evaluate rouge_score

In [ ]:
GROQ_API_KEY = "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"

groq_client = Groq(api_key=GROQ_API_KEY)
MODEL_NAME = "llama-3.1-8b-instant"

def call_llm(prompt):
    response = groq_client.chat.completions.create(
        model=MODEL_NAME,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.3
    )
    return response.choices[0].message.content

In [ ]:
pipe = StableDiffusionPipeline.from_pretrained(
    "runwayml/stable-diffusion-v1-5",
    torch_dtype=torch.float16
).to("cuda")

In [ ]:
chat_agent = Agent(
    role="AI Tutor",
    goal="Provide interactive tutoring responses",
    backstory="Expert educator across STEM and humanities.",
    verbose=False
)

lesson_agent = Agent(
    role="Lesson Planning Specialist",
    goal="Generate structured lesson plans",
    backstory="Curriculum design expert.",
    verbose=False
)

project_agent = Agent(
    role="Project Mentor",
    goal="Generate project ideas and execution steps",
    backstory="Hands-on practical learning specialist.",
    verbose=False
)

In [ ]:
def reflex_router(mode, user_input):

    if mode == "Chat":
        prompt = f"""
        Provide an engaging tutoring explanation for:
        {user_input}

        Use:
        - Clear structure
        - Examples
        - Summary section
        """

        return call_llm(prompt)

    elif mode == "Lesson Planner":
        prompt = f"""
        Create a professional lesson plan for:
        {user_input}

        Include:
        - Objectives
        - Learning outcomes
        - Timeline
        - Activities
        - Assessment
        """

        return call_llm(prompt)

    elif mode == "Project Lists":
        prompt = f"""
        Generate project ideas for:
        {user_input}

        Include:
        - Beginner, Intermediate, Advanced
        - Tools required
        - Deliverables
        """

        return call_llm(prompt)

    elif mode == "Text to Image":
        image = pipe(user_input).images[0]
        return image

In [ ]:
rouge = evaluate.load("rouge")

def evaluate_response(output_text):
    reference = "Structured response with objectives, explanation, and summary."
    scores = rouge.compute(
        predictions=[output_text],
        references=[reference]
    )
    return scores

In [ ]:
custom_css = """
body {
    background-color: #f4f7fb;
}
.banner {
    white-space: nowrap;
    overflow: hidden;
}
.banner span {
    display: inline-block;
    padding-left: 100%;
    animation: marquee 15s linear infinite;
    font-weight: bold;
    color: #0b3d91;
}
@keyframes marquee {
    0% { transform: translateX(0); }
    100% { transform: translateX(-100%); }
}
"""

with gr.Blocks(css=custom_css, theme=gr.themes.Soft()) as demo:

    gr.HTML("""
    <div class="banner">
        <span>AI Tutor by Collins Aerospace</span>
    </div>
    """)

    gr.Markdown("## Intelligent AI Tutor Platform")

    mode = gr.Radio(
        ["Chat", "Lesson Planner", "Project Lists", "Text to Image"],
        label="Select Mode"
    )

    user_input = gr.Textbox(
        label="Enter Topic / Prompt"
    )

    run_button = gr.Button("Generate", variant="primary")

    output_text = gr.Markdown()
    output_image = gr.Image()
    metrics_box = gr.JSON()

    def run_system(mode, user_input):

        result = reflex_router(mode, user_input)

        if mode == "Text to Image":
            return None, result, None

        metrics = evaluate_response(result)
        return result, None, metrics

    run_button.click(
        run_system,
        inputs=[mode, user_input],
        outputs=[output_text, output_image, metrics_box]
    )

demo.launch(debug=True)